# Toronto Public Library - Exploratory Data Analysis (EDA)

This notebook performs Exploratory Data Analysis (EDA) on various datasets from the City of Toronto's Open Data portal related to the Toronto Public Library (TPL). The goal is to understand historical performance, identify trends, and provide insights for managerial assessment.

Visualizations include:
- Time series charts of `Circulation`, `Events`, `Registrations`, `Sessions` values
- Scatter plot of values vs `SquareFootage`
- Rankings of Branches by different values

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
cwd = os.environ["REPO_ROOT"]
os.chdir(cwd)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import re

# --- Configuration ---
# Set plot style for better aesthetics
sns.set_style("whitegrid")

# Define paths to the datasets
BRANCH_INFO_PATH = "data/library-branch-general-information.csv"
CIRCULATION_PATH = "data/library-circulation.csv"
VISITS_PATH = "data/library-visits.csv"
REGISTRATIONS_PATH = "data/library-card-registrations.csv"
WORKSTATION_USAGE_PATH = "data/library-workstation-usage.csv"
EVENTS_PATH = "data/library-branch-programs-and-events-feed.csv"

## 1. Data Loading and Initial Inspection

This section loads all relevant datasets into pandas DataFrames. It also performs initial checks like displaying head, info, describe, and missing values.

In [5]:
print("--- 1. Data Loading and Initial Inspection ---")

# Load datasets
branch_info = pd.read_csv(BRANCH_INFO_PATH)
circulation = pd.read_csv(CIRCULATION_PATH)
visits = pd.read_csv(VISITS_PATH)
registrations = pd.read_csv(REGISTRATIONS_PATH)
workstation_usage = pd.read_csv(WORKSTATION_USAGE_PATH)
events = pd.read_csv(EVENTS_PATH)

# Drop '_id' column from usage and events dataframes as it's just an index and causes merge issues
# This ensures clean merges later and removes redundant information.
circulation = circulation.drop(columns=['_id'])
visits = visits.drop(columns=['_id'])
registrations = registrations.drop(columns=['_id'])
workstation_usage = workstation_usage.drop(columns=['_id'])
events = events.drop(columns=['_id'])

# Display initial information for each DataFrame
print("--- Branch Information (branch_info) ---")
print("Head:", branch_info.head())
print("Info:")
branch_info.info()
print("Description:", branch_info.describe())
print("Missing Values:", branch_info.isnull().sum())

print("--- Circulation Data (circulation) ---")
print("Head:", circulation.head())
print("Info:")
circulation.info()
print("Description:", circulation.describe())
print("Missing Values:", circulation.isnull().sum())

print("--- Visits Data (visits) ---")
print("Head:", visits.head())
print("Info:")
visits.info()
print("Description:", visits.describe())
print("Missing Values:", visits.isnull().sum())

print("--- Registrations Data (registrations) ---")
print("Head:", registrations.head())
print("Info:")
registrations.info()
print("Description:", registrations.describe())
print("Missing Values:", registrations.isnull().sum())

print("--- Workstation Usage Data (workstation_usage) ---")
print("Head:", workstation_usage.head())
print("Info:")
workstation_usage.info()
print("Description:", workstation_usage.describe())
print("Missing Values:", workstation_usage.isnull().sum())

print("--- Events Data (events) ---")
print("Head:", events.head())
print("Info:")
events.info()
print("Description:", events.describe())
print("Missing Values:", events.isnull().sum())

--- 1. Data Loading and Initial Inspection ---
--- Branch Information (branch_info) ---
Head:    _id BranchCode  PhysicalBranch       BranchName  \
0    1         AB               1           Albion   
1    2        ACD               1  Albert Campbell   
2    3         AD               1        Alderwood   
3    4         AG               1        Agincourt   
4    5         AH               1   Armour Heights   

                                     Address PostalCode  \
0     1515 Albion Road, Toronto, ON, M9V 1B2    M9V 1B2   
1  496 Birchmount Road, Toronto, ON, M1K 1N8    M1K 1N8   
2      2 Orianna Drive, Toronto, ON, M8W 4Y1    M8W 4Y1   
3     155 Bonis Avenue, Toronto, ON, M1T 3W6    M1T 3W6   
4     2140 Avenue Road, Toronto, ON, M5M 4M7    M5M 4M7   

                             Website     Telephone SquareFootage  \
0          https://www.tpl.ca/albion  416-394-5170         29000   
1  https://www.tpl.ca/albertcampbell  416-396-8890         28957   
2       https://www.tp

## 2. Overall Library Usage Trends

This section analyzes and visualizes the high-level trends in TPL's performance across all branches for key metrics over time.

In [6]:
print("--- 2. Overall Library Usage Trends ---")

# Merge all usage dataframes into a single dataframe based on 'Year' and 'BranchCode'.
# Using 'outer' merge to ensure all years and branches are included, even if some data is missing.
usage_data_merged = circulation.merge(visits, on=['Year', 'BranchCode'], how='outer') \
                        .merge(registrations, on=['Year', 'BranchCode'], how='outer') \
                        .merge(workstation_usage, on=['Year', 'BranchCode'], how='outer')

# Fill NaN values with 0 for numerical columns after merging.
# This is important for accurate summation of usage metrics.
for col in ['Circulation', 'Visits', 'Registrations', 'Sessions']:
    usage_data_merged[col] = usage_data_merged[col].fillna(0)

# Aggregate yearly totals for all metrics.
# This provides a summary of TPL's overall performance evolution.
yearly_summary = usage_data_merged.groupby('Year')[['Circulation', 'Visits', 'Registrations', 'Sessions']].sum().reset_index()

print("--- Yearly Summary of Library Usage ---")
print(yearly_summary)

# Plotting overall trends using Plotly Express for interactivity.
# These line plots show how each key metric has changed year over year.
fig_circ = px.line(yearly_summary, x='Year', y='Circulation', title='Total Circulation Over Time')
fig_circ.show()

fig_visits = px.line(yearly_summary, x='Year', y='Visits', title='Total Visits Over Time')
fig_visits.show()

fig_reg = px.line(yearly_summary, x='Year', y='Registrations', title='Total Registrations Over Time')
fig_reg.show()

fig_ws = px.line(yearly_summary, x='Year', y='Sessions', title='Total Workstation Sessions Over Time')
fig_ws.show()

--- 2. Overall Library Usage Trends ---
--- Yearly Summary of Library Usage ---
    Year  Circulation      Visits  Registrations   Sessions
0   2012   32032036.0  18872613.0       141310.0  6138192.0
1   2013   32145021.0  18485394.0       145343.0  6465748.0
2   2014   32034795.0  18335931.0       153041.0  6537286.0
3   2015   32505963.0  18153077.0       151417.0  6692432.0
4   2016   31910577.0  18232367.0       155950.0  6467319.0
5   2017   30098890.0  17370032.0       157050.0  5715251.0
6   2018   30555570.0  17577373.0       198405.0        0.0
7   2019   30662033.0  17248761.0       202732.0        0.0
8   2020   21000916.0   5557751.0        69946.0        0.0
9   2021   24229094.0   4029488.0       114379.0        0.0
10  2022   26598932.0   9566486.0       186425.0        0.0
11  2023   24992899.0  12507823.0       251891.0        0.0


## 3. Branch-Specific Performance Analysis

This section focuses on the performance of individual library branches. It helps identify top/bottom performers and potential correlations with branch attributes.

In [7]:
print("--- 3. Branch-Specific Performance Analysis ---")

# Merge all data into a single DataFrame for comprehensive analysis.
# This combines usage statistics with detailed branch information.
full_data = usage_data_merged.merge(branch_info, on='BranchCode', how='left')

# Calculate total performance metrics per branch across all years.
# 'first' is used for 'SquareFootage' and 'ServiceTier' as they are static attributes per branch.
branch_performance = full_data.groupby('BranchName').agg({
    'Circulation': 'sum',
    'Visits': 'sum',
    'Registrations': 'sum',
    'Sessions': 'sum',
    'SquareFootage': 'first',
    'ServiceTier': 'first'
}).reset_index()

print("--- Branch Performance Summary (All Years) ---")
print(branch_performance.head())

# Sort and display top/bottom branches for each metric.
# This highlights branches with exceptional or struggling performance.
print("--- Top 10 Branches by Total Circulation ---")
print(branch_performance.nlargest(10, 'Circulation'))

print("--- Bottom 10 Branches by Total Circulation ---")
print(branch_performance.nsmallest(10, 'Circulation'))

print("--- Top 10 Branches by Total Visits ---")
print(branch_performance.nlargest(10, 'Visits'))

print("--- Bottom 10 Branches by Total Visits ---")
print(branch_performance.nsmallest(10, 'Visits'))

print("--- Top 10 Branches by Total Registrations ---")
print(branch_performance.nlargest(10, 'Registrations'))

print("--- Bottom 10 Branches by Total Registrations ---")
print(branch_performance.nsmallest(10, 'Registrations'))

print("--- Top 10 Branches by Total Workstation Sessions ---")
print(branch_performance.nlargest(10, 'Sessions'))

print("--- Bottom 10 Branches by Total Workstation Sessions ---")
print(branch_performance.nsmallest(10, 'Sessions'))

# Visualizations for branch performance using Plotly Express.
# Bar charts for top 15 branches provide a quick visual comparison.
fig_branch_circ = px.bar(branch_performance.nlargest(15, 'Circulation'),
                         x='Circulation', y='BranchName', orientation='h',
                         title='Top 15 Branches by Total Circulation (All Years)')
fig_branch_circ.show()

fig_branch_visits = px.bar(branch_performance.nlargest(15, 'Visits'),
                         x='Visits', y='BranchName', orientation='h',
                         title='Top 15 Branches by Total Visits (All Years)')
fig_branch_visits.show()

fig_branch_reg = px.bar(branch_performance.nlargest(15, 'Registrations'),
                        x='Registrations', y='BranchName', orientation='h',
                        title='Top 15 Branches by Total Registrations (All Years)')
fig_branch_reg.show()

fig_branch_ws = px.bar(branch_performance.nlargest(15, 'Sessions'),
                       x='Sessions', y='BranchName', orientation='h',
                       title='Top 15 Branches by Total Workstation Sessions (All Years)')
fig_branch_ws.show()

# Scatter plots to explore correlations between branch attributes and usage.
# This helps understand if factors like size influence performance.
fig_sqft_circ = px.scatter(branch_performance, x='SquareFootage', y='Circulation',
                           hover_name='BranchName', title='Circulation vs. Square Footage')
fig_sqft_circ.show()

fig_sqft_visits = px.scatter(branch_performance, x='SquareFootage', y='Visits',
                            hover_name='BranchName', title='Visits vs. Square Footage')
fig_sqft_visits.show()

--- 3. Branch-Specific Performance Analysis ---
--- Branch Performance Summary (All Years) ---
        BranchName  Circulation     Visits  Registrations  Sessions  \
0        Agincourt    7709220.0  3512745.0        33961.0  883719.0   
1  Albert Campbell    3126548.0  2446043.0        17970.0  860846.0   
2           Albion    3623347.0  3269625.0        60295.0  834377.0   
3        Alderwood    1585933.0   944173.0         6708.0   97268.0   
4    Amesbury Park    1089998.0   522344.0         9818.0   95601.0   

  SquareFootage ServiceTier  
0         27000          DL  
1         28957          DL  
2         29000          DL  
3          7341          NL  
4          6320          NL  
--- Top 10 Branches by Total Circulation ---
                     BranchName  Circulation      Visits  Registrations  \
105             Virtual Library   81563119.0         0.0       105075.0   
73   North York Central Library   12663452.0  12949589.0       122760.0   
0                     Aginco

## 4. Programs and Events Analysis

This section analyzes the types of programs and events offered, their distribution, and trends over time to understand engagement and offerings.

In [8]:
print("--- 4. Programs and Events Analysis ---")

# Clean event descriptions by removing HTML tags for better readability.
events['description'] = events['description'].apply(lambda x: re.sub(r'<[^>]*>', '', str(x)))

# Convert 'startdate' to datetime objects for time-based analysis.
# 'errors='coerce'' will turn unparseable dates into NaT (Not a Time).
events['startdate'] = pd.to_datetime(events['startdate'], errors='coerce')

# Analyze top event types based on the three event type columns.
# This helps identify the most common categories of programs offered.
print("--- Top 10 Event Types (eventtype1) ---")
print(events['eventtype1'].value_counts().nlargest(10))

print("--- Top 10 Event Types (eventtype2) ---")
print(events['eventtype2'].value_counts().nlargest(10))

print("--- Top 10 Event Types (eventtype3) ---")
print(events['eventtype3'].value_counts().nlargest(10))

# Analyze top age groups targeted by events.
# This provides insight into who the programs are designed for.
print("--- Top 5 Age Groups (agegroup1) ---")
print(events['agegroup1'].value_counts().nlargest(5))

# Analyze the number of events per library.
# This shows which branches are most active in terms of program offerings.
print("--- Top 10 Libraries by Number of Events ---")
print(events['library'].value_counts().nlargest(10))

# Plotting top event types using Plotly Express.
fig_event_types = px.bar(events['eventtype1'].value_counts().nlargest(10).reset_index(),
                         x='count', y='eventtype1', orientation='h',
                         title='Top 10 Event Types (eventtype1)')
fig_event_types.show()

# Plotting top age groups using Plotly Express.
fig_age_groups = px.bar(events['agegroup1'].value_counts().nlargest(5).reset_index(),
                        x='count', y='agegroup1', orientation='h',
                        title='Top 5 Age Groups for Events')
fig_age_groups.show()

# Plotting events per library using Plotly Express.
fig_events_per_library = px.bar(events['library'].value_counts().nlargest(10).reset_index(),
                                x='count', y='library', orientation='h',
                                title='Top 10 Libraries by Number of Events')
fig_events_per_library.show()

# Calculate and plot the monthly trend of events.
# This helps identify seasonality or changes in event frequency over time.
# First, extract year and month to create a period.
events['month_year'] = events['startdate'].dt.to_period('M')
# Count events per month and sort by time.
monthly_events = events['month_year'].value_counts().sort_index().reset_index()
monthly_events.columns = ['Month_Year', 'Event_Count']
# Convert Month_Year back to datetime for plotting
monthly_events['Month_Year'] = monthly_events['Month_Year'].dt.to_timestamp()

fig_monthly_events = px.line(monthly_events, x='Month_Year', y='Event_Count',
                             title='Monthly Trend of Events')
fig_monthly_events.show()

--- 4. Programs and Events Analysis ---
--- Top 10 Event Types (eventtype1) ---
eventtype1
00-Hobbies Crafts & Games           966
00-Reading Programs & Storytimes    850
00-Computer & Library Training      381
00-Book Clubs & Writers Groups      350
00-After School                     300
01-Ready for Reading Storytimes     288
00-Culture Arts & Entertainment     270
00-ESL & Newcomer Programs          218
00-Science & Technology             164
00-Health & Wellness                154
Name: count, dtype: int64
--- Top 10 Event Types (eventtype2) ---
eventtype2
00-Hobbies Crafts & Games           209
01-Ready for Reading Storytimes     191
00-Reading Programs & Storytimes    189
00-Culture Arts & Entertainment     122
01-After School Club                121
01-Pop-Up Learning Labs             107
00-Health & Wellness                 81
00-After School                      65
00-Science & Technology              53
01-TPL Teens                         46
Name: count, dtype: int64
--- To